# AIAdvisor

# Does not work - need fixes

1. Read all report pages in a directory
2. Send them to Pinecone
3. Read Pinecone index
4. For a query, find relevant documents
5. Using Langchain, send the query and relevant documents to ChatGPT
6. Get the answer

In [6]:
import openai
import os
from langchain.vectorstores import Pinecone as Pinecone_lch
from langchain.chains.question_answering import load_qa_chain
from aiadvisor_pinecone_lch import get_answer
from dotenv import load_dotenv, find_dotenv
import pinecone
from aiadvisor_pinecone_lch import index_name
import global_config
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')

In [8]:
# Mark owns this case id in production
case_id = "458"
namespace = case_id
# initialize pinecone
pinecone.init(
    api_key=os.getenv('PINECONE_API_KEY'),     
    environment="us-west4-gcp"  # next to api key in console
)
pinecone.describe_index(index_name)
index_pc = pinecone.Index("casify")
#index_pc.delete(deleteAll = True, namespace = namespace)
description = pinecone.describe_index(index_name)
print(description)

IndexDescription(name='casify', metric='cosine', replicas=1, dimension=1536.0, shards=1, pods=1, pod_type='p1.x1', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')


In [9]:
def split_docs(documents, chunk_size=1000, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(documents)
    return docs

In [10]:
#loader = TextLoader("../data/159/processed/pages/courtdoc_page_1.txt")
#document = loader.load()

In [11]:
#documents = [document]

In [12]:
#docs = split_docs(documents)

In [13]:
Pinecone_lch.from_documents(docs, embeddings, index_name=index_name, namespace=namespace)

NameError: name 'docs' is not defined

In [14]:
from langchain.document_loaders import DirectoryLoader

directory = pages_dir

def load_docs(directory):
  loader = DirectoryLoader(directory)
  documents = loader.load()
  return documents

documents = load_docs(directory)
len(documents)

NameError: name 'pages_dir' is not defined

https://python.langchain.com/en/latest/modules/indexes/text_splitters/getting_started.html

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=1000,chunk_overlap=20):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
  docs = text_splitter.split_documents(documents)
  return docs

docs = split_docs(documents)
print(len(docs))

AttributeError: 'list' object has no attribute 'page_content'

### Print example

In [ ]:
print(docs[0].page_content)
print(len(docs))

In [ ]:
import openai
from langchain.embeddings.openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)

query_result = embeddings.embed_query("Hello world")
len(query_result)

https://python.langchain.com/en/latest/modules/indexes/vectorstores/examples/pinecone.html

In [ ]:
# write all data to index_lch
index_lch = Pinecone_lch.from_documents(docs, embeddings, index_name=index_name, namespace = namespace)

# if you already have an index, you can load it like this
index_lch = Pinecone_lch.from_existing_index(index_name, embeddings, namespace = namespace)
index_pc = pinecone.Index(index_name)
index_stats_response = index_pc.describe_index_stats()
print(index_stats_response)

In [ ]:
def get_similar_docs(query,num_sources=5,score=False):
  if score:
    similar_docs = index_lch.similarity_search_with_score(query,k=num_sources)
  else:
    similar_docs = index_lch.similarity_search(query,k=num_sources)
  return similar_docs


In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(model_name=MODEL)

chain = load_qa_chain(llm, chain_type="stuff")

https://python.langchain.com/en/latest/use_cases/question_answering.html

In [ ]:
answer=get_answer("What is the charge?", "159")
print(answer)